In [39]:
import networkx as nx
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [56]:

df1 = pd.read_json('../data/openalex_authors_complete.json')

In [41]:
df2 = pd.read_json('../data/openalex_data.json')

In [57]:
df1= df1.T
df1.isnull().sum()

id                               0
orcid                        19008
display_name                     0
display_name_alternatives        0
works_count                      0
cited_by_count                   0
summary_stats                    0
ids                              0
affiliations                     6
last_known_institutions          6
topics                           6
topic_share                      6
x_concepts                       0
counts_by_year                   0
works_api_url                    0
updated_date                     0
created_date                     0
dtype: int64

In [ ]:
df1[df1['display_name']=='Suilán Estévez-Velarde']

,id,orcid,display_name,display_name_alternatives,works_count,cited_by_count,summary_stats,ids,affiliations,last_known_institutions,topics,topic_share,x_concepts,counts_by_year,works_api_url,updated_date,created_date
A5010783785,https://openalex.org/A5010783785,https://orcid.org/0000-0001-6707-1442,Suilán Estévez-Velarde,"[Suilán Estévez‐Velarde, Suilan Estévez‐Velard...",33,147,"{'2yr_mean_citedness': 2.0, 'h_index': 7, 'i10...",{'openalex': 'https://openalex.org/A5010783785...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I2656126', 'ror'...","[{'id': 'https://openalex.org/T10028', 'displa...","[{'id': 'https://openalex.org/T12380', 'displa...","[{'id': 'https://openalex.org/C41008148', 'wik...","[{'year': 2024, 'works_count': 1, 'cited_by_co...",https://api.openalex.org/works?filter=author.i...,2025-05-18T04:59:22.124264,2023-07-21


In [44]:
df1['orcid'] = df1['orcid'].str.split('https://orcid.org/').str[1]

In [45]:

print("Total de autores:", len(df1))
print("ORCIDs únicos:", df1['orcid'].nunique())
print("ORCIDs no nulos:", df1['orcid'].notna().sum())

duplicados_orcid = df1[df1['orcid'].duplicated(keep=False) & df1['orcid'].notna()]
print(f"\nAutores con ORCID duplicado: {len(duplicados_orcid)}")

if len(duplicados_orcid) > 0:
    print("\nORCIDs duplicados:")
    orcids_duplicados = duplicados_orcid['orcid'].value_counts()
    print(orcids_duplicados)

Total de autores: 30199
ORCIDs únicos: 11191
ORCIDs no nulos: 11191

Autores con ORCID duplicado: 0


In [59]:

print("=== ANÁLISIS DE DISPLAY NAMES ===")
print("Total de autores:", len(df1))
print("Display names únicos:", df1['display_name'].nunique())
print("Display names no nulos:", df1['display_name'].notna().sum())

def normalize_display_name(name):
    if not isinstance(name, str):
        return name
    
    replacements = {
        'á': 'a', 'é': 'e', 'í': 'i', 'ó': 'o', 'ú': 'u',
        'ü': 'u', 'ñ': 'n', 
        'à': 'a', 'è': 'e', 'ì': 'i', 'ò': 'o', 'ù': 'u',
        'â': 'a', 'ê': 'e', 'î': 'i', 'ô': 'o', 'û': 'u',
        'ã': 'a', 'õ': 'o', 
        'ä': 'a', 'ë': 'e', 'ï': 'i', 'ö': 'o', 'ü': 'u',
        'ç': 'c',
        '-': ' ', '.': ' ', ',': ' ', ';': ' ', ':': ' '
    }
    

    name = name.lower()
    
    for old, new in replacements.items():
        name = name.replace(old, new)
    
    name = ' '.join(name.split())
    
    return name

df1['display_name'] = df1['display_name'].apply(normalize_display_name)

duplicados_display = df1[df1['display_name'].duplicated(keep=False) & df1['display_name'].notna()]
print(f"\nAutores con display name duplicado: {len(duplicados_display)}")

if len(duplicados_display) > 0:
    print("\nDisplay names duplicados (top 20):")
    display_duplicados = duplicados_display['display_name'].value_counts().head(20)
    print(display_duplicados)
else:
    print("No se encontraron duplicados de display name")

=== ANÁLISIS DE DISPLAY NAMES ===
Total de autores: 30199
Display names únicos: 29665
Display names no nulos: 30199

Autores con display name duplicado: 2799

Display names duplicados (top 20):
display_name
jing li                    7
f dionisio                 7
deleted author             6
manuel ortiz               6
r lalana                   6
j a rodriguez              5
j martinez                 5
r rodriguez                5
maryuri garcia gonzalez    5
carlos rodriguez           5
r gonzalez                 5
m lee                      4
m alvarez                  4
jose gonzalez              4
a garcia                   4
jorge martinez             4
j perez                    4
jose luis                  4
d perez                    4
c rodriguez                4
Name: count, dtype: int64


In [60]:

def get_author_country(institutions):
    """Extrae el código de país de las instituciones de un autor"""
    if not isinstance(institutions, list) or len(institutions) == 0:
        return None
    
    for inst in institutions:
        if isinstance(inst, dict) and inst.get('country_code'):
            return inst['country_code']
    return None


print("Agregando columna de país...")
df1['country'] = df1['last_known_institutions'].apply(get_author_country)


print("\nDistribución de países (top 10):")
country_counts = df1['country'].value_counts().head(10)
print(country_counts)

print(f"\nAutores sin información de país: {df1['country'].isna().sum()}")
print(f"Países únicos: {df1['country'].nunique()}")

Agregando columna de país...

Distribución de países (top 10):
country
CU    11170
ES     2429
MX     2231
US     1823
BR     1691
EC      796
DE      662
FR      566
IT      500
CL      411
Name: count, dtype: int64

Autores sin información de país: 2927
Países únicos: 158


In [61]:

print("\n=== IDENTIFICANDO CANDIDATOS PARA CONSOLIDACIÓN ===")


df1['name_country_key'] = df1['display_name'] + '_' + df1['country'].fillna('UNKNOWN')

duplicados_mismo_pais = df1[df1['name_country_key'].duplicated(keep=False) & df1['country'].notna()]
print(f"Autores con mismo nombre y país: {len(duplicados_mismo_pais)}")

if len(duplicados_mismo_pais) > 0:
    grupos_consolidacion = duplicados_mismo_pais.groupby(['display_name', 'country']).size().sort_values(ascending=False)
    print(f"\nGrupos a consolidar: {len(grupos_consolidacion)}")
    print("\nTop 10 grupos con más duplicados:")
    print(grupos_consolidacion.head(10))
    
    print("\n=== EJEMPLOS DE CONSOLIDACIÓN ===")
    for i, ((nombre, pais), count) in enumerate(grupos_consolidacion.head(3).items()):
        print(f"\nGrupo {i+1}: '{nombre}' en {pais} ({count} autores)")
        grupo = df1[(df1['display_name'] == nombre) & (df1['country'] == pais)]
        for j, (idx, autor) in enumerate(grupo.iterrows()):
            print(f"  Autor {j+1}: ID={idx}, ORCID={autor.get('orcid', 'N/A')}, Works={autor.get('works_count', 'N/A')}")
else:
    print("No se encontraron autores duplicados del mismo país")


=== IDENTIFICANDO CANDIDATOS PARA CONSOLIDACIÓN ===
Autores con mismo nombre y país: 1466

Grupos a consolidar: 713

Top 10 grupos con más duplicados:
display_name                           country
maryuri garcia gonzalez                CU         5
e silva                                CU         4
manuel ortiz                           MX         4
louis charles de menorval              FR         4
o quintela                             CU         4
julio ivan gonzalez piedra             CU         4
suilan estevez velarde                 CU         4
jing li                                CN         4
ivan gonzalez gongora                  CU         3
cristina lopez calleja hiort lorenzen  CU         3
dtype: int64

=== EJEMPLOS DE CONSOLIDACIÓN ===

Grupo 1: 'maryuri garcia gonzalez' en CU (5 autores)
  Autor 1: ID=A5043412018, ORCID=None, Works=63
  Autor 2: ID=A5043362609, ORCID=None, Works=7
  Autor 3: ID=A5114119738, ORCID=None, Works=1
  Autor 4: ID=A5063339172, ORCID=http

In [62]:

def consolidate_authors(group):
    """Consolida un grupo de autores con el mismo nombre y país"""

    group_sorted = group.copy()
    group_sorted['has_orcid'] = group_sorted['orcid'].notna()
    group_sorted['works_count_safe'] = group_sorted['works_count'].fillna(0)
    
    group_sorted = group_sorted.sort_values([
        'has_orcid', 'works_count_safe', 'id'
    ], ascending=[False, False, True])
    
    principal = group_sorted.iloc[0]
    others = group_sorted.iloc[1:]
    
    consolidated = principal.copy()
    
    consolidated['works_count'] = group['works_count'].fillna(0).sum()
    consolidated['cited_by_count'] = group['cited_by_count'].fillna(0).sum()
    
    all_alternatives = []
    for alt in group['display_name_alternatives'].dropna():
        if isinstance(alt, list):
            all_alternatives.extend(alt)
        elif isinstance(alt, str):
            all_alternatives.append(alt)
    
    for _, other in others.iterrows():
        if other['display_name'] not in all_alternatives:
            all_alternatives.append(other['display_name'])
    
    consolidated['display_name_alternatives'] = list(set(all_alternatives)) if all_alternatives else None
    
    if pd.isna(consolidated['orcid']):
        for _, other in others.iterrows():
            if pd.notna(other['orcid']):
                consolidated['orcid'] = other['orcid']
                break
    
    all_institutions = []
    for institutions in group['last_known_institutions'].dropna():
        if isinstance(institutions, list):
            all_institutions.extend(institutions)
    
    unique_institutions = []
    seen_ids = set()
    for inst in all_institutions:
        if isinstance(inst, dict) and inst.get('id') not in seen_ids:
            unique_institutions.append(inst)
            seen_ids.add(inst.get('id'))
    
    consolidated['last_known_institutions'] = unique_institutions if unique_institutions else None
    
    consolidated['consolidated_author_ids'] = others.index.tolist()
    
    return consolidated, others.index.tolist()


print("\n=== REALIZANDO CONSOLIDACIÓN ===")
df_consolidated = df1.copy()
removed_authors = []
consolidation_log = []

for (nombre, pais), group in duplicados_mismo_pais.groupby(['display_name', 'country']):
    if len(group) > 1:  
        consolidated_author, removed_ids = consolidate_authors(group)
        
        principal_id = consolidated_author.name
        df_consolidated.loc[principal_id] = consolidated_author
        
        removed_authors.extend(removed_ids)
        
        consolidation_log.append({
            'name': nombre,
            'country': pais,
            'principal_id': principal_id,
            'removed_ids': removed_ids,
            'total_authors': len(group),
            'total_works': consolidated_author['works_count']
        })

df_consolidated = df_consolidated.drop(removed_authors)

print(f"Autores eliminados: {len(removed_authors)}")
print(f"Grupos consolidados: {len(consolidation_log)}")
print(f"Autores antes: {len(df1)}")
print(f"Autores después: {len(df_consolidated)}")
print(f"Reducción: {len(df1) - len(df_consolidated)} autores")


=== REALIZANDO CONSOLIDACIÓN ===
Autores eliminados: 753
Grupos consolidados: 713
Autores antes: 30199
Autores después: 29446
Reducción: 753 autores


In [63]:
df1 = df_consolidated.copy()
print("\n✓ Variable df1 actualizada con datos consolidados")
print(f"df1 ahora tiene {len(df1):,} autores únicos")


✓ Variable df1 actualizada con datos consolidados
df1 ahora tiene 29,446 autores únicos


In [64]:

df1['old_ids'] = None

for entry in consolidation_log:
    principal_id = entry['principal_id']
    removed_ids = entry['removed_ids']
    
    if principal_id in df1.index:
        df1.at[principal_id, 'old_ids'] = removed_ids

df1['old_ids'] = df1['old_ids'].apply(lambda x: x if x else None)
            

In [70]:
df1[df1['display_name']=='alejandro piad morffis'.lower()]

,id,orcid,display_name,display_name_alternatives,works_count,cited_by_count,summary_stats,ids,affiliations,last_known_institutions,topics,topic_share,x_concepts,counts_by_year,works_api_url,updated_date,created_date,country,name_country_key,old_ids
A5025376903,https://openalex.org/A5025376903,https://orcid.org/0000-0001-9522-3239,alejandro piad morffis,"[Alejandro Piad Morffis, alejandro piad morffi...",37,168,"{'2yr_mean_citedness': 0.6666666666666661, 'h_...",{'openalex': 'https://openalex.org/A5025376903...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I2656126', 'ror'...","[{'id': 'https://openalex.org/T10028', 'displa...","[{'id': 'https://openalex.org/T12380', 'displa...","[{'id': 'https://openalex.org/C41008148', 'wik...","[{'year': 2024, 'works_count': 0, 'cited_by_co...",https://api.openalex.org/works?filter=author.i...,2025-05-20T15:51:29.786965,2023-07-21,CU,alejandro piad morffis_CU,[A5026879236]
